# Векторизация текста: IMDB Reviews

Задача: бинарная классификация тональности отзывов (positive / negative).

План:
1. Загрузка и очистка данных
2. Train / test split
3. Векторизация: CountVectorizer и TF-IDF
4. Обучение логистической регрессии и сравнение
5. Самые «позитивные» и «негативные» слова

In [22]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

## 1. Загрузка данных

In [23]:
df = pd.read_csv("IMDB Dataset (2).csv")
print(df.shape)
print("Дубликатов:", df.duplicated().sum())
print(df["sentiment"].value_counts())
df.head()

(50000, 2)
Дубликатов: 418
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [24]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(49582, 2)

## 2. Очистка текста

В ~58% отзывов есть HTML-теги `<br />` — убираем их, оставляем только буквы и приводим к нижнему регистру.

In [25]:
def clean(text):
    text = re.sub(r"<.*?>", " ", text)          # убираем <br /> и прочий HTML
    text = re.sub(r"[^a-zA-Z']", " ", text)     # оставляем только буквы
    return text.lower()

df["clean"] = df["review"].apply(clean)
y = (df["sentiment"] == "positive").astype(int)

print(df["review"][1][:300])
print("---")
print(df["clean"][1][:300])

A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" 
---
a wonderful little production    the filming technique is very unassuming  very old time bbc fashion and gives a comforting  and sometimes discomforting  sense of realism to the entire piece    the actors are extremely well chosen  michael sheen not only  has got all the polari  but he has all the v


## 3. Train / test split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], y, test_size=0.2, random_state=42, stratify=y
)
len(X_train), len(X_test)

(39665, 9917)

## 4. Векторизация и обучение

Важно: `fit` векторизатора — **только на train**, к test применяем только `transform`, чтобы не было утечки данных.

Сравним несколько вариантов:
- **CountVectorizer** — просто количество слов
- **TF-IDF** — слова, взвешенные по редкости
- TF-IDF **со стоп-словами** и **без** (в стандартном списке есть `not`, `no` — важные для тональности)

In [27]:
def evaluate(name, vec):
    X_tr = vec.fit_transform(X_train)
    X_te = vec.transform(X_test)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_train)
    acc = accuracy_score(y_test, clf.predict(X_te))
    print(f"{name:30s} shape={X_tr.shape}  accuracy={acc:.4f}")
    return vec, clf, acc

common = dict(max_features=50000, ngram_range=(1, 2), min_df=2)

results = {}
results["Count"]                  = evaluate("Count",                  CountVectorizer(**common))
results["TF-IDF + stop_words"]    = evaluate("TF-IDF + stop_words",    TfidfVectorizer(**common, stop_words="english"))
results["TF-IDF без stop_words"]  = evaluate("TF-IDF без stop_words",  TfidfVectorizer(**common))

Count                          shape=(39665, 50000)  accuracy=0.8996
TF-IDF + stop_words            shape=(39665, 50000)  accuracy=0.8963
TF-IDF без stop_words          shape=(39665, 50000)  accuracy=0.9054


## 5. Подробный отчёт по лучшей модели

In [28]:
best_name = max(results, key=lambda k: results[k][2])
vec, clf, _ = results[best_name]
print("Лучшая:", best_name)
print(classification_report(y_test, clf.predict(vec.transform(X_test)),
                            target_names=["negative", "positive"]))

Лучшая: TF-IDF без stop_words
              precision    recall  f1-score   support

    negative       0.92      0.89      0.90      4940
    positive       0.90      0.92      0.91      4977

    accuracy                           0.91      9917
   macro avg       0.91      0.91      0.91      9917
weighted avg       0.91      0.91      0.91      9917



## 6. Какие слова модель считает важными

Коэффициенты логистической регрессии: большой положительный → слово «тянет» к positive, отрицательный → к negative.

In [29]:
words = vec.get_feature_names_out()
coefs = clf.coef_[0]
order = np.argsort(coefs)

top = pd.DataFrame({
    "negative": words[order[:20]],
    "neg_coef": coefs[order[:20]].round(2),
    "positive": words[order[::-1][:20]],
    "pos_coef": coefs[order[::-1][:20]].round(2),
})
top

,negative,neg_coef,positive,pos_coef
0,bad,-9.11,great,8.61
1,worst,-8.50,excellent,6.53
2,awful,-7.49,perfect,5.18
3,boring,-6.78,wonderful,4.86
4,the worst,-6.65,amazing,4.67
5,poor,-6.32,best,4.46
6,terrible,-5.95,the best,4.18
7,waste,-5.83,loved,4.10
8,nothing,-5.26,brilliant,4.10
9,worse,-4.72,fun,4.09


## 7. Проверка на своих текстах

In [30]:
my_reviews = [
    "This movie was absolutely wonderful, great acting!",
    "Boring plot, terrible acting, waste of time.",
    "The film was not good at all.",
]
probs = clf.predict_proba(vec.transform([clean(t) for t in my_reviews]))[:, 1]
for t, p in zip(my_reviews, probs):
    print(f"{p:.2f}  {'positive' if p > 0.5 else 'negative'}  | {t}")

0.93  positive  | This movie was absolutely wonderful, great acting!
0.00  negative  | Boring plot, terrible acting, waste of time.
0.05  negative  | The film was not good at all.


---
# Часть 2. Трансформер (sentence-transformers)

## 8. Идея: чем трансформер отличается от TF-IDF

**TF-IDF** — это «мешок слов»: каждое слово/биграмма — отдельная колонка, порядок и смысл предложения не учитываются. Поэтому в *«I wouldn't recommend this movie»* слово `recommend` тянуло в плюс, и модель выдала 0.60.

**Трансформер** — нейросеть, заранее обученная на огромном количестве текстов. Ключевое отличие — механизм **attention (внимания)**: при обработке каждого слова модель «смотрит» на все остальные слова предложения. Поэтому вектор слова `recommend` в *«I **wouldn't** recommend»* и в *«I **highly** recommend»* получается **разным** — он зависит от контекста.

### Как мы будем использовать трансформер

Есть два подхода:

| Подход | Что делаем | Плюсы / минусы |
|---|---|---|
| **Feature extraction** (наш) | Берём готовую модель «как есть», получаем вектор каждого отзыва, сверху обучаем обычную LogisticRegression | Быстро, не нужен GPU, просто. Модель не подстраивается под нашу задачу |
| **Fine-tuning** | Дообучаем сам трансформер на наших отзывах | Лучшее качество (~93–95%), но долго и нужен GPU |

То есть схема та же, что в части 1, — меняется **только способ получить вектор**:

```
TF-IDF:       текст → TfidfVectorizer       → вектор 50 000 (разреженный) → LogisticRegression
Трансформер:  текст → SentenceTransformer   → вектор 384 (плотный)        → LogisticRegression
```

## 9. Выбор модели

Используем **`all-MiniLM-L6-v2`** из библиотеки `sentence-transformers`:
- маленькая и быстрая: 6 слоёв, ~22 млн параметров (BERT-base — 110 млн);
- обучена специально так, чтобы **похожие по смыслу предложения получали близкие векторы**;
- на выходе — вектор из **384 чисел** для любого текста.

При первом запуске модель (~90 МБ) скачается из интернета в `~/.cache/huggingface`, дальше берётся из кэша.

**Устройство (device):** на Mac с чипом M1/M2/M3 есть GPU, в PyTorch он называется `mps`. На нём кодирование в несколько раз быстрее, чем на CPU.

In [31]:
import time
import torch
from sentence_transformers import SentenceTransformer

if torch.backends.mps.is_available():
    device = "mps"        # GPU на Mac (Apple Silicon)
elif torch.cuda.is_available():
    device = "cuda"       # GPU NVIDIA
else:
    device = "cpu"
print("device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print(model)

device: mps


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2633.41it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


`print(model)` показывает, из чего модель состоит — это конвейер из трёх шагов:

1. **Transformer** — сама нейросеть (BERT-подобная). На вход — токены, на выход — по вектору 384 на **каждый токен**. `max_seq_length: 256` — длиннее 256 токенов текст обрезается.
2. **Pooling** (`pooling_mode_mean_tokens: True`) — из векторов всех токенов делаем **один** вектор текста, просто усредняя их.
3. **Normalize** — приводим вектор к длине 1 (удобно для сравнения векторов через косинусное сходство).

Разберём каждый шаг подробнее.

## 10. Шаг 1: подготовка текста

Для трансформера предобработка **другая**, чем для TF-IDF:

| | TF-IDF | Трансформер |
|---|---|---|
| HTML-теги `<br />` | убираем | убираем (это мусор) |
| нижний регистр | да | **нет** — у модели свой токенизатор, он сам разберётся |
| пунктуация | убираем | **оставляем** — `!`, `?`, `n't` несут смысл |
| стоп-слова, лемматизация | можно | **нет** — модели нужен естественный текст, `not` и `wouldn't` критически важны |

Модель обучалась на обычном, «живом» тексте — и лучше всего работает именно на нём.

In [32]:
def clean_for_bert(text):
    text = re.sub(r"<.*?>", " ", text)        # только убираем HTML
    text = re.sub(r"\s+", " ", text)          # и лишние пробелы
    return text.strip()

df["bert_text"] = df["review"].apply(clean_for_bert)

# Берём ТЕ ЖЕ train/test, что и в части 1 (по индексам), чтобы сравнение было честным
bert_train = df.loc[X_train.index, "bert_text"].tolist()
bert_test  = df.loc[X_test.index,  "bert_text"].tolist()
print(len(bert_train), len(bert_test))
print(bert_train[0][:300])

39665 9917
I am one of Jehovah's Witnesses and I also work in an acute care medical facility. Over the years I have seen people die from hemolytic reactions to blood transfusions, have attended numerous conferences on blood born pathogens, and have seen several patients become seriously ill from pathogens indu


## 11. Шаг 2: токенизация

Трансформер не работает со словами напрямую. Сначала **токенизатор** режет текст на **токены** — это слова или **кусочки слов** (subword-токенизация, алгоритм WordPiece).

Зачем кусочки: словарь модели ограничен (~30 000 токенов). Редкое слово разбивается на знакомые части, например `unwatchable` → `un`, `##watch`, `##able` (`##` значит «продолжение слова»). Так модель может обработать **любое** слово, даже которого не видела.

Также добавляются служебные токены:
- `[CLS]` — в начале текста,
- `[SEP]` — в конце.

Каждый токен затем заменяется на его номер в словаре (`input_ids`) — именно эти числа идут в нейросеть.

In [33]:
tokenizer = model.tokenizer

example = "I wouldn't recommend this movie to anyone. Unwatchable!"
enc = tokenizer(example)

print("Токены:   ", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("input_ids:", enc["input_ids"])

Токены:    ['[CLS]', 'i', 'wouldn', "'", 't', 'recommend', 'this', 'movie', 'to', 'anyone', '.', 'un', '##watch', '##able', '!', '[SEP]']
input_ids: [101, 1045, 2876, 1005, 1056, 16755, 2023, 3185, 2000, 3087, 1012, 4895, 18866, 3085, 999, 102]


Обратите внимание: `wouldn't` превратилось в `wouldn`, `'`, `t` — но, в отличие от TF-IDF, модель видит **все** эти токены **в контексте** соседних, и `t` не выбрасывается.

### Длина отзывов и обрезка

У модели ограничение — **256 токенов**. Посмотрим, сколько отзывов длиннее и будут обрезаны (берётся только начало текста).

In [34]:
sample = df["bert_text"].sample(2000, random_state=42)
n_tokens = sample.apply(lambda t: len(tokenizer(t)["input_ids"]))

print(n_tokens.describe().round(0))
print(f"\nДоля отзывов длиннее 256 токенов: {(n_tokens > 256).mean():.0%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (350 > 256). Running this sequence through the model will result in indexing errors


count    2000.0
mean      303.0
std       223.0
min        22.0
25%       162.0
50%       227.0
75%       376.0
max      2084.0
Name: bert_text, dtype: float64

Доля отзывов длиннее 256 токенов: 43%


Примерно у трети-половины отзывов модель увидит только начало. Это ограничение подхода, но обычно тональность понятна уже из первых предложений. (Можно было бы резать длинный отзыв на куски и усреднять их векторы — оставим как идею для улучшения.)

## 12. Шаг 3: от токенов к вектору текста

Что происходит внутри `model.encode(text)`:

```
"I wouldn't recommend..."
        │  токенизатор
        ▼
[CLS] i wouldn ' t recommend ... [SEP]          ← N токенов
        │  6 слоёв трансформера (attention: каждый токен «смотрит» на все остальные)
        ▼
N векторов по 384 числа                          ← у каждого токена свой, КОНТЕКСТНЫЙ вектор
        │  mean pooling (среднее по токенам)
        ▼
1 вектор из 384 чисел                            ← вектор всего текста
        │  нормализация
        ▼
вектор длины 1
```

Проверим, что вектора действительно отражают **смысл**. Сравним предложения через **косинусное сходство** (1 — одинаковый смысл, около 0 — не связаны). Векторы нормализованы, поэтому косинусное сходство = скалярное произведение.

In [35]:
sentences = [
    "I wouldn't recommend this movie to anyone.",
    "I highly recommend this movie to everyone.",
    "This film is a complete waste of time.",
    "One of the best films I have ever seen.",
]
emb = model.encode(sentences, normalize_embeddings=True)
print("Форма:", emb.shape)   # 4 предложения × 384 числа

sim = emb @ emb.T            # матрица косинусных сходств
pd.DataFrame(sim.round(2), index=[s[:35] for s in sentences], columns=[s[:20] for s in sentences])

Форма: (4, 384)


,I wouldn't recommend,I highly recommend t,This film is a compl,One of the best film
I wouldn't recommend this movie to,1.00,0.83,0.52,0.59
I highly recommend this movie to ev,0.83,1.00,0.50,0.69
This film is a complete waste of ti,0.52,0.50,1.00,0.50
One of the best films I have ever s,0.59,0.69,0.50,1.00


Важный момент: модель `all-MiniLM-L6-v2` обучалась искать **похожие по теме** тексты, а не различать тональность. Поэтому *«wouldn't recommend»* и *«highly recommend»* могут оказаться довольно близкими — оба про рекомендацию фильма.

Но это не проблема: в 384 числах информация о тональности всё равно есть, и **логистическая регрессия поверх** научится её вытаскивать — найдёт нужные «направления» в пространстве векторов.

## 13. Шаг 4: кодируем весь датасет

Теперь прогоняем все ~50 000 отзывов через модель. Это самая долгая часть:
- `batch_size=64` — отзывы подаются в модель пачками по 64 (так быстрее, чем по одному);
- на Mac M-серии (`mps`) это занимает порядка **5–10 минут**, на CPU — дольше.

Чтобы не ждать каждый раз, **сохраняем векторы в файл** `.npy` и при повторном запуске просто загружаем их.

In [36]:
import os

EMB_FILE = "embeddings_minilm.npy"

if os.path.exists(EMB_FILE):
    embeddings = np.load(EMB_FILE)
    print("Загружено из файла")
else:
    start = time.time()
    embeddings = model.encode(
        df["bert_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(EMB_FILE, embeddings)
    print(f"Готово за {(time.time() - start) / 60:.1f} мин")

print(embeddings.shape)   # (кол-во отзывов, 384)

Загружено из файла
(49582, 384)


Каждая строка — отзыв, каждый столбец — одно из 384 «скрытых» свойств текста. В отличие от TF-IDF, у столбцов **нет понятных названий** (не «слово worst», а что-то абстрактное), зато матрица маленькая и плотная: 384 столбца вместо 50 000.

Заметьте: здесь **нет утечки данных**, хотя мы закодировали весь датасет сразу. Модель уже обучена и ничего не узнаёт из наших данных — она просто применяется к каждому тексту по отдельности (в отличие от TfidfVectorizer, который при `fit` считает статистику по корпусу).

## 14. Шаг 5: обучаем классификатор на векторах

Разбиваем векторы по тем же индексам train/test и обучаем ту же LogisticRegression, что и в части 1.

In [37]:
E_train = embeddings[X_train.index]
E_test  = embeddings[X_test.index]

clf_bert = LogisticRegression(max_iter=2000, C=1.0)
clf_bert.fit(E_train, y_train)

pred_bert = clf_bert.predict(E_test)
acc_bert = accuracy_score(y_test, pred_bert)

print(f"TF-IDF (лучшая из части 1): {results[best_name][2]:.4f}")
print(f"MiniLM + LogReg:            {acc_bert:.4f}")
print()
print(classification_report(y_test, pred_bert, target_names=["negative", "positive"]))

TF-IDF (лучшая из части 1): 0.9054
MiniLM + LogReg:            0.8163

              precision    recall  f1-score   support

    negative       0.82      0.81      0.81      4940
    positive       0.81      0.82      0.82      4977

    accuracy                           0.82      9917
   macro avg       0.82      0.82      0.82      9917
weighted avg       0.82      0.82      0.82      9917



### Как интерпретировать результат

Не удивляйтесь, если accuracy окажется **около или даже ниже** TF-IDF. Это частая ситуация, и вот почему:
- **TF-IDF «заточен» под наши данные**: 50 000 признаков, выученных именно на отзывах IMDB, а классификатор сам решает, какие слова важны.
- **MiniLM не обучался на тональности** — он сжимает текст в 384 числа «общего смысла», часть информации о тональности при этом теряется.
- **Обрезка до 256 токенов** — для длинных отзывов модель видит только начало.

Зато трансформер должен лучше справляться с **отрицаниями и контекстом** — проверим на сложных примерах.

## 15. Шаг 6: проверка на сложных примерах

In [38]:
tricky = [
    "This movie was absolutely wonderful, great acting!",
    "Boring plot, terrible acting, waste of time.",
    "The film was not good at all.",
    "I wouldn't recommend this movie to anyone.",
    "I expected it to be terrible, but it was actually great.",
    "Not bad at all, I enjoyed it.",
]

p_tfidf = clf.predict_proba(vec.transform([clean(t) for t in tricky]))[:, 1]
p_bert  = clf_bert.predict_proba(model.encode(tricky, normalize_embeddings=True))[:, 1]

pd.DataFrame({"отзыв": tricky, "TF-IDF": p_tfidf.round(2), "MiniLM": p_bert.round(2)})

,отзыв,TF-IDF,MiniLM
0,"This movie was absolutely wonderful, great act...",0.93,1.00
1,"Boring plot, terrible acting, waste of time.",0.00,0.00
2,The film was not good at all.,0.05,0.00
3,I wouldn't recommend this movie to anyone.,0.60,0.18
4,"I expected it to be terrible, but it was actua...",0.50,0.08
5,"Not bad at all, I enjoyed it.",0.39,0.91


Числа — вероятность класса **positive**. Смотрим, на каких примерах модели расходятся: обычно это отрицания (*not bad*, *wouldn't recommend*) и контраст (*expected terrible, but great*).

## 16. Бонус: модель, дообученная на тональность (fine-tuning)

Выше мы использовали «универсальный» трансформер + свой классификатор. Посмотрим, как работает модель, которую **уже дообучили** именно на определение тональности — `distilbert-base-uncased-finetuned-sst-2-english` (дообучена на отзывах о фильмах SST-2).

Здесь не нужен ни векторизатор, ни LogisticRegression: модель сразу выдаёт метку и уверенность. Это и есть результат подхода **fine-tuning** из таблицы в разделе 8.

*(При первом запуске скачается ~270 МБ.)*

In [39]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

out = sentiment(tricky)
p_ft = [o["score"] if o["label"] == "POSITIVE" else 1 - o["score"] for o in out]

pd.DataFrame({
    "отзыв": tricky,
    "TF-IDF": p_tfidf.round(2),
    "MiniLM": p_bert.round(2),
    "DistilBERT-SST2": np.round(p_ft, 2),
})

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6072.95it/s]


,отзыв,TF-IDF,MiniLM,DistilBERT-SST2
0,"This movie was absolutely wonderful, great act...",0.93,1.00,1.00
1,"Boring plot, terrible acting, waste of time.",0.00,0.00,0.00
2,The film was not good at all.,0.05,0.00,0.00
3,I wouldn't recommend this movie to anyone.,0.60,0.18,0.96
4,"I expected it to be terrible, but it was actua...",0.50,0.08,1.00
5,"Not bad at all, I enjoyed it.",0.39,0.91,1.00


In [ ]:
# Оценим её на части нашего теста (на всём тесте — несколько минут)
sub = np.random.RandomState(42).choice(len(bert_test), 2000, replace=False)
texts_sub = [bert_test[i] for i in sub]

out = sentiment(texts_sub, batch_size=32, truncation=True, max_length=512)
pred_ft = np.array([1 if o["label"] == "POSITIVE" else 0 for o in out])

y_sub = y_test.values[sub]
print(f"На тех же 2000 отзывах:")
print(f"  TF-IDF:          {accuracy_score(y_sub, clf.predict(vec.transform(X_test.iloc[sub]))):.4f}")
print(f"  MiniLM + LogReg: {accuracy_score(y_sub, pred_bert[sub]):.4f}")
print(f"  DistilBERT-SST2: {accuracy_score(y_sub, pred_ft):.4f}   ← без обучения на наших данных!")

---
# Часть 3. Fine-tuning: дообучаем трансформер на наших данных

## 17. Идея

В разделе 16 модель, дообученная на **чужих** данных (SST-2), перенесла их «перекосы»: фраза *«This movie is to anyone.»* получала 1.00 positive. Теперь дообучим трансформер **сами — на отзывах IMDB**.

Чем это отличается от части 2 (MiniLM + LogReg):

| | Feature extraction (часть 2) | Fine-tuning (часть 3) |
|---|---|---|
| Веса трансформера | **заморожены** | **обучаются** вместе с классификатором |
| Что учится | только LogisticRegression (385 чисел) | все ~67 млн параметров DistilBERT |
| Attention | смотрит «как привык» | учится смотреть на то, что важно **для тональности** |

Берём **`distilbert-base-uncased`** — «чистый» DistilBERT, который прошёл только предобучение (угадывание слов) и **ничего не знает про тональность**. Всё знание о тональности он получит из наших данных.

**Почему только часть train:** на Mac один шаг обучения ~1 сек, весь train (~40 000 отзывов) занял бы больше часа на эпоху. Возьмём **4 000 отзывов** — это ~12 минут на 2 эпохи. Интересно посмотреть, хватит ли этого, чтобы догнать TF-IDF, обученный на всех 40 000.

## 17.1. Модель и токенизатор

`AutoModelForSequenceClassification` = DistilBERT + **голова классификации** сверху:

```
текст → DistilBERT (6 слоёв) → вектор токена [CLS] (768 чисел) → Linear → Linear → 2 числа (logits)
```

- **Тело** (DistilBERT) — загружается с предобученными весами.
- **Голова** (два слоя Linear) — создаётся **со случайными весами**, её не было в предобученной модели. Поэтому при загрузке появится предупреждение `MISSING: classifier.weight ...` — это нормально, так и должно быть.
- Вектор берётся у служебного токена **`[CLS]`**: в процессе обучения модель научится «собирать» в него смысл всего текста через attention.
- На выходе 2 числа — **logits** — «сырые» оценки для классов negative и positive. Softmax превращает их в вероятности.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

torch.manual_seed(42)

MODEL_NAME = "distilbert-base-uncased"
ft_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ft_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

n_params = sum(p.numel() for p in ft_model.parameters())
print(f"Параметров: {n_params / 1e6:.0f} млн")

In [ ]:
# Проверим: до обучения модель гадает наугад (голова случайная) — вероятности около 0.5
ft_model.eval()
with torch.no_grad():
    e = ft_tokenizer(tricky[:3], padding=True, return_tensors="pt").to(device)
    print(torch.softmax(ft_model(**e).logits, dim=-1)[:, 1].cpu().numpy().round(2))

## 17.2. Данные для обучения

1. Берём случайные 4 000 отзывов из **train** (тест не трогаем — на нём будем честно проверять).
2. Токенизируем:
   - `truncation=True, max_length=256` — обрезаем длинные отзывы до 256 токенов (как в MiniLM);
   - `padding="max_length"` — короткие **дополняем** токеном `[PAD]` до 256, чтобы все примеры в батче имели одинаковую длину (нейросеть работает с прямоугольными тензорами);
   - `attention_mask` — маска из 1 и 0: 1 = настоящий токен, 0 = `[PAD]`. Attention **игнорирует** позиции с 0, чтобы паддинг не влиял на смысл.
3. `DataLoader` нарезает данные на **батчи по 16** и перемешивает их каждую эпоху (`shuffle=True`).

In [ ]:
N_TRAIN = 4000
MAX_LEN = 256

rng = np.random.RandomState(42)
train_idx = rng.choice(len(bert_train), N_TRAIN, replace=False)
ft_texts  = [bert_train[i] for i in train_idx]
ft_labels = y_train.values[train_idx]
print("Баланс классов:", np.bincount(ft_labels))

enc = ft_tokenizer(ft_texts, truncation=True, max_length=MAX_LEN,
                   padding="max_length", return_tensors="pt")
print("input_ids:     ", enc["input_ids"].shape)
print("attention_mask:", enc["attention_mask"][0][-10:])   # конец первого отзыва

train_ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(ft_labels))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
print("Батчей за эпоху:", len(train_loader))

## 17.3. Оптимизатор и расписание learning rate

- **AdamW** — стандартный оптимизатор для трансформеров.
- **`lr=2e-5`** — learning rate очень маленький (в обычных нейросетях часто 1e-3). Модель уже многое знает, и большой шаг «сломал» бы предобученные веса — это называется **catastrophic forgetting**. Мы лишь слегка подстраиваем их.
- **`weight_decay=0.01`** — небольшая регуляризация против переобучения.
- **Warmup + linear decay**: первые 10% шагов lr плавно растёт от 0 до 2e-5 (чтобы случайная голова не «дёрнула» тело модели сильными градиентами в начале), потом линейно падает до 0.

```
lr
 ▲   /\
 │  /   \
 │ /      \
 │/         \___
 └──────────────▶ шаги
 warmup  decay
```

In [ ]:
EPOCHS = 2

optimizer = AdamW(ft_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print("Всего шагов:", total_steps)

## 17.4. Цикл обучения

Один **шаг** обучения на батче из 16 отзывов:

1. **Forward**: прогоняем батч через модель. Т.к. передаём `labels`, модель сама считает **loss** (cross-entropy — насколько предсказанные вероятности далеки от правильных ответов).
2. **Backward** (`loss.backward()`): считаем градиенты — в какую сторону подвинуть **каждый** из 67 млн весов, чтобы loss уменьшился.
3. **Gradient clipping**: если градиенты слишком большие, обрезаем их норму до 1 — защита от резких «скачков».
4. **`optimizer.step()`**: двигаем веса. **`scheduler.step()`**: обновляем lr. **`zero_grad()`**: обнуляем градиенты перед следующим шагом.

**Эпоха** = один проход по всем 4 000 отзывам (250 шагов).

`ft_model.train()` включает режим обучения (в нём работает dropout — случайное «выключение» нейронов для регуляризации).

Следим за loss: в начале около **0.69** (= ln 2, модель гадает 50/50), должен падать.

⏱ ~6 минут на эпоху на Mac M-серии (весь цикл ~12 минут).

In [ ]:
history = []
start = time.time()

for epoch in range(EPOCHS):
    ft_model.train()
    running = 0.0
    for step, (ids, mask, labels) in enumerate(train_loader):
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        out = ft_model(input_ids=ids, attention_mask=mask, labels=labels)   # 1. forward + loss
        out.loss.backward()                                                 # 2. градиенты
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)          # 3. clipping
        optimizer.step()                                                    # 4. шаг
        scheduler.step()
        optimizer.zero_grad()

        history.append(out.loss.item())
        running += out.loss.item()
        if (step + 1) % 50 == 0:
            print(f"эпоха {epoch + 1}  шаг {step + 1}/{len(train_loader)}  "
                  f"loss (ср. за 50 шагов) = {running / 50:.3f}  "
                  f"[{(time.time() - start) / 60:.1f} мин]")
            running = 0.0

print(f"\nОбучение заняло {(time.time() - start) / 60:.1f} мин")

In [ ]:
import matplotlib.pyplot as plt

smooth = pd.Series(history).rolling(20).mean()
plt.figure(figsize=(9, 3.5))
plt.plot(history, alpha=0.3, label="loss на батче")
plt.plot(smooth, label="скользящее среднее (20)")
plt.axhline(np.log(2), color="gray", ls="--", label="ln 2 — случайное угадывание")
plt.xlabel("шаг"); plt.ylabel("loss"); plt.legend(); plt.title("Обучение DistilBERT")
plt.show()

## 17.5. Оценка

`ft_model.eval()` — режим предсказания (dropout выключен), `torch.no_grad()` — не считаем градиенты (быстрее и меньше памяти).

Здесь используем `padding=True` (а не `"max_length"`): дополняем только до длины самого длинного отзыва **в батче** — так быстрее.

Сравниваем на **тех же 2000 тестовых отзывах**, что и в разделе 16.

In [ ]:
def predict_proba_ft(texts, batch_size=64):
    ft_model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            e = ft_tokenizer(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                             padding=True, return_tensors="pt").to(device)
            logits = ft_model(**e).logits
            probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

p_myft = predict_proba_ft([bert_test[i] for i in sub])
pred_myft = (p_myft > 0.5).astype(int)

print("На тех же 2000 тестовых отзывах:")
print(f"  TF-IDF (40 000 отзывов):          {accuracy_score(y_sub, clf.predict(vec.transform(X_test.iloc[sub]))):.4f}")
print(f"  MiniLM + LogReg:                  {accuracy_score(y_sub, pred_bert[sub]):.4f}")
print(f"  DistilBERT-SST2 (чужие данные):   {accuracy_score(y_sub, pred_ft):.4f}")
print(f"  DistilBERT, наш fine-tune (4 000): {accuracy_score(y_sub, pred_myft):.4f}")
print()
print(classification_report(y_sub, pred_myft, target_names=["negative", "positive"]))

In [ ]:
tricky2 = tricky + [
    "I can't recommend this movie to anyone.",
    "This movie is to anyone.",
]
p_ft2 = [o["score"] if o["label"] == "POSITIVE" else 1 - o["score"] for o in sentiment(tricky2)]

pd.DataFrame({
    "отзыв": tricky2,
    "TF-IDF": clf.predict_proba(vec.transform([clean(t) for t in tricky2]))[:, 1].round(2),
    "MiniLM": clf_bert.predict_proba(model.encode(tricky2, normalize_embeddings=True))[:, 1].round(2),
    "SST-2": np.round(p_ft2, 2),
    "наш fine-tune": predict_proba_ft(tricky2).round(2),
})

## 17.6. Сохранение модели

Дообученную модель стоит сохранить, чтобы не обучать заново. Потом её можно загрузить одной строкой (и даже использовать через `pipeline`, как SST-2):

```python
ft_model = AutoModelForSequenceClassification.from_pretrained("distilbert_imdb").to(device)
ft_tokenizer = AutoTokenizer.from_pretrained("distilbert_imdb")
```

In [ ]:
ft_model.save_pretrained("distilbert_imdb")
ft_tokenizer.save_pretrained("distilbert_imdb")

---
# Часть 4. Word2Vec

## 18. Идея

По «эволюции» методов Word2Vec стоит **между** TF-IDF и трансформерами:

| | TF-IDF | **Word2Vec** | Трансформер |
|---|---|---|---|
| Вектор слова | нет (слово = номер колонки) | **плотный, ~100 чисел, есть смысл** | плотный, зависит от контекста |
| «good» и «great» | никак не связаны | **близкие векторы** | близкие векторы |
| «recommend» в разных фразах | одна колонка | **один и тот же вектор** | разные векторы |

**Главная идея Word2Vec** (Миколов, 2013): *«слово определяется своим окружением»*. Если «awful» и «terrible» встречаются в похожих контекстах (*«the acting was ___»*), их векторы должны быть похожими.

### Как обучается

Модель — крошечная нейросеть (один скрытый слой). Берём каждое слово текста и его соседей в окне `window` слов:

```
"the acting was absolutely terrible and boring"
                         ↑ центральное слово: terrible, window=2
               соседи: was, absolutely, and, boring
```

- **Skip-gram** (`sg=1`): по центральному слову угадываем соседей: `terrible → was, absolutely, and, boring`.
- **CBOW** (`sg=0`): наоборот, по соседям угадываем центральное слово. Быстрее, но хуже работает с редкими словами.

Сами угадывания нам не нужны. Нужны **веса скрытого слоя**: строка весов для каждого слова — и есть его вектор. Чтобы хорошо угадывать соседей, сеть вынуждена давать словам с похожими соседями похожие векторы.

## 18.1. Подготовка данных

Word2Vec принимает **список предложений, где каждое предложение — список слов** (токенов). Используем текст `clean` из части 1 (нижний регистр, только буквы) и просто режем по пробелам.

Стоп-слова **не удаляем**: для Word2Vec они часть контекста («not», «very» важны для соседей).

Обучаем **только на train**: тест модель не должна видеть, даже без меток.

In [54]:
from gensim.models import Word2Vec

tok_train = [t.split() for t in X_train]
tok_test  = [t.split() for t in X_test]

print(len(tok_train), "отзывов")
print(tok_train[0][:20])

39665 отзывов
['i', 'am', 'one', 'of', "jehovah's", 'witnesses', 'and', 'i', 'also', 'work', 'in', 'an', 'acute', 'care', 'medical', 'facility', 'over', 'the', 'years', 'i']


## 18.2. Обучение Word2Vec

Параметры:
- **`vector_size=100`** — длина вектора слова. Обычно 100–300.
- **`window=5`** — сколько соседей слева и справа считаем контекстом.
- **`min_count=5`** — слова, встретившиеся меньше 5 раз, выбрасываем (опечатки, редкие имена; для них не хватит контекстов, чтобы выучить хороший вектор).
- **`sg=1`** — skip-gram (лучше для редких слов).
- **`epochs=5`** — сколько раз проходим по всему корпусу.
- **`workers=4`** — потоки процессора.

⏱ ~1–3 минуты.

⚠️ Может появиться несколько строк `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`. Это безобидный артефакт сборки gensim на Mac, на результат не влияет.

In [55]:
start = time.time()
w2v = Word2Vec(
    sentences=tok_train,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,
    epochs=5,
    workers=4,
    seed=42,
)
print(f"Обучено за {time.time() - start:.0f} сек")
print("Слов в словаре:", len(w2v.wv))
print("Вектор слова 'good':", w2v.wv["good"][:8].round(2), "...", w2v.wv["good"].shape)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Обучено за 37 сек
Слов в словаре: 37837
Вектор слова 'good': [ 0.18 -0.1   0.03  0.12  0.09 -0.36 -0.09  0.13] ... (100,)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


## 18.3. Что выучила модель

`w2v.wv` (word vectors) хранит векторы всех слов. Самое интересное — посмотреть **ближайших соседей** по косинусному сходству.

In [56]:
for word in ["awful", "boring", "recommend", "brilliant", "good"]:
    neighbors = w2v.wv.most_similar(word, topn=6)
    print(f"{word:10s} →", ", ".join(f"{w} ({s:.2f})" for w, s in neighbors))

awful      → terrible (0.88), horrible (0.85), dreadful (0.85), horrid (0.84), atrocious (0.82), abysmal (0.80)
boring     → dull (0.92), pointless (0.86), tedious (0.85), uneventful (0.84), unoriginal (0.81), predictable (0.80)
recommend  → recomend (0.83), advise (0.78), recommended (0.75), reccomend (0.75), suggest (0.73), heartily (0.72)
brilliant  → fantastic (0.83), superb (0.79), terrific (0.78), wonderful (0.78), excellent (0.77), great (0.76)
good       → decent (0.81), great (0.77), bad (0.76), serviceable (0.72), nice (0.71), solid (0.71)


Обратите внимание:
- **синонимы** находятся отлично: `awful → terrible, horrible, dreadful`;
- модель выучила даже **опечатки**: `recommend → recomend, reccomend`, потому что они стоят в тех же контекстах;
- **но `good` и `bad` оказываются близкими!** Это главная слабость Word2Vec для анализа тональности. «The movie was **good**» и «The movie was **bad**» имеют одинаковое окружение, поэтому векторы похожи. Word2Vec учит **сходство по употреблению**, а не по тональности.

In [57]:
print("good ~ bad:  ", round(float(w2v.wv.similarity("good", "bad")), 2))
print("good ~ great:", round(float(w2v.wv.similarity("good", "great")), 2))

# Знаменитая «арифметика слов»: king - man + woman ≈ ?
print(w2v.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3))

good ~ bad:   0.76
good ~ great: 0.77
[('queen', 0.6369879841804504), ('anne', 0.6274283528327942), ('carmen', 0.5981206297874451)]


## 18.4. От векторов слов к вектору отзыва

Word2Vec даёт вектор **каждого слова**, а классификатору нужен **один вектор на отзыв**. Простейший способ — **усреднить** векторы всех слов отзыва (слова не из словаря пропускаем):

```
"the acting was terrible"  →  mean( v(the), v(acting), v(was), v(terrible) )  →  вектор из 100 чисел
```

Минус: при усреднении теряются **порядок слов** и **отрицания**. «not good» превращается в среднее между «not» и «good». А в длинном отзыве из 300 слов одно важное слово «размывается» среди сотен нейтральных.

In [58]:
def doc_vector(tokens, wv):
    vecs = [wv[w] for w in tokens if w in wv]          # только слова из словаря
    if not vecs:
        return np.zeros(wv.vector_size)                 # если ни одного — нулевой вектор
    return np.mean(vecs, axis=0)

W_train = np.vstack([doc_vector(t, w2v.wv) for t in tok_train])
W_test  = np.vstack([doc_vector(t, w2v.wv) for t in tok_test])
print(W_train.shape, W_test.shape)

(39665, 100) (9917, 100)


## 18.5. Классификатор

In [59]:
clf_w2v = LogisticRegression(max_iter=2000)
clf_w2v.fit(W_train, y_train)
acc_w2v = accuracy_score(y_test, clf_w2v.predict(W_test))

print(f"TF-IDF:              {results[best_name][2]:.4f}")
print(f"Word2Vec (среднее):  {acc_w2v:.4f}")
print(f"MiniLM + LogReg:     {acc_bert:.4f}")

TF-IDF:              0.9054
Word2Vec (среднее):  0.8654
MiniLM + LogReg:     0.8163


## 18.6. Улучшение: взвешенное среднее по IDF

Идея: при усреднении давать **больший вес редким, информативным словам** («masterpiece», «unwatchable») и меньший — частым («the», «movie»). Вес слова = его **IDF** (из TF-IDF).

Спойлер: на этом датасете помогает слабо или не помогает вовсе. Логистическая регрессия и так учится игнорировать «шумные» направления в векторе. Но это стандартный приём, и его полезно знать.

In [60]:
idf_vec = TfidfVectorizer(min_df=5, token_pattern=r"[a-z']+")   # тот же способ деления на слова, что и split()
idf_vec.fit(X_train)
idf = dict(zip(idf_vec.get_feature_names_out(), idf_vec.idf_))

def doc_vector_idf(tokens, wv):
    words = [w for w in tokens if w in wv and w in idf]
    if not words:
        return np.zeros(wv.vector_size)
    return np.average([wv[w] for w in words], axis=0, weights=[idf[w] for w in words])

Wi_train = np.vstack([doc_vector_idf(t, w2v.wv) for t in tok_train])
Wi_test  = np.vstack([doc_vector_idf(t, w2v.wv) for t in tok_test])

clf_w2v_idf = LogisticRegression(max_iter=2000).fit(Wi_train, y_train)
acc_w2v_idf = accuracy_score(y_test, clf_w2v_idf.predict(Wi_test))
print(f"Word2Vec (среднее):          {acc_w2v:.4f}")
print(f"Word2Vec (среднее по IDF):   {acc_w2v_idf:.4f}")

Word2Vec (среднее):          0.8654
Word2Vec (среднее по IDF):   0.8624


## 18.7. Предобученные векторы: GloVe

Вместо своего Word2Vec можно взять векторы, **уже обученные на огромном корпусе**. **GloVe** — родственный Word2Vec метод (Stanford, 2014), версия `glove-wiki-gigaword-100` обучена на Википедии и новостях: 400 000 слов, по 100 чисел.

Загружаем с Hugging Face (~130 МБ при первом запуске).

*Почему не через `gensim.downloader`: Python, установленный с python.org, на Mac по умолчанию не видит системные SSL-сертификаты, и загрузка падает с `CERTIFICATE_VERIFY_FAILED`. Hugging Face использует свои сертификаты, поэтому с ним такой проблемы нет.*

In [ ]:
import os
from huggingface_hub import snapshot_download
from gensim.models import KeyedVectors

glove_path = snapshot_download("fse/glove-wiki-gigaword-100")
glove = KeyedVectors.load(os.path.join(glove_path, "glove-wiki-gigaword-100.model"))
print("Слов:", len(glove), " размерность:", glove.vector_size)

for word in ["awful", "boring", "recommend"]:
    print(f"{word:10s} →", ", ".join(w for w, _ in glove.most_similar(word, topn=6)))

In [ ]:
G_train = np.vstack([doc_vector(t, glove) for t in tok_train])
G_test  = np.vstack([doc_vector(t, glove) for t in tok_test])

clf_glove = LogisticRegression(max_iter=2000).fit(G_train, y_train)
acc_glove = accuracy_score(y_test, clf_glove.predict(G_test))

print(f"Word2Vec, свой (IMDB):     {acc_w2v:.4f}")
print(f"GloVe, предобученный:      {acc_glove:.4f}")

Свой Word2Vec выигрывает у GloVe, хотя GloVe обучен на гораздо большем корпусе. Причина — **домен**: GloVe учился на Википедии и новостях, где мало эмоциональной лексики о фильмах. Наш Word2Vec учился ровно на тех текстах, которые классифицируем.

## 18.8. Сложные примеры

In [ ]:
def w2v_proba(texts):
    return clf_w2v.predict_proba(np.vstack([doc_vector(clean(t).split(), w2v.wv) for t in texts]))[:, 1]

pd.DataFrame({
    "отзыв": tricky2,
    "TF-IDF": clf.predict_proba(vec.transform([clean(t) for t in tricky2]))[:, 1].round(2),
    "Word2Vec": w2v_proba(tricky2).round(2),
    "MiniLM": clf_bert.predict_proba(model.encode(tricky2, normalize_embeddings=True))[:, 1].round(2),
    "наш fine-tune": predict_proba_ft(tricky2).round(2),
})

,отзыв,TF-IDF,Word2Vec,MiniLM,наш fine-tune
0,"This movie was absolutely wonderful, great act...",0.93,0.99,1.00,0.98
1,"Boring plot, terrible acting, waste of time.",0.00,0.00,0.00,0.03
2,The film was not good at all.,0.05,0.67,0.00,0.07
3,I wouldn't recommend this movie to anyone.,0.60,0.75,0.18,0.13
4,"I expected it to be terrible, but it was actua...",0.50,0.30,0.08,0.96
5,"Not bad at all, I enjoyed it.",0.39,0.87,0.91,0.90
6,I can't recommend this movie to anyone.,0.74,0.69,0.53,0.06
7,This movie is to anyone.,0.43,0.10,0.96,0.73


: 

Смотрите на примеры с отрицаниями: Word2Vec с усреднением **не понимает порядок слов**. Для него «not good» — это просто набор {not, good}, примерно как и для TF-IDF без биграмм.

## 19. Выводы

| Метод | Как получается вектор | Плюсы | Минусы |
|---|---|---|---|
| **TF-IDF** | счёт слов/биграмм | быстро, интерпретируемо, сильный бейзлайн | не понимает контекст и отрицания |
| **Word2Vec (среднее)** | средний вектор слов, 100 чисел | понимает синонимы, компактный, быстро обучается | теряет порядок слов и отрицания; good ≈ bad |
| **MiniLM + LogReg** | готовый трансформер, 384 числа | понимает смысл, компактный вектор | не обучен на тональность, обрезка 256 токенов, медленнее |
| **Fine-tuned трансформер** | трансформер, дообученный на задаче | понимает контекст и отрицания | долго обучать, нужен GPU, «чёрный ящик», переносит перекосы обучающих данных |

Результаты на одних и тех же 2000 тестовых отзывах:

| Модель | Обучающих отзывов | Accuracy |
|---|---|---|
| TF-IDF + LogReg | 40 000 | 0.913 |
| DistilBERT, наш fine-tune | **4 000** | 0.888 |
| DistilBERT-SST2 | 0 (чужие данные) | 0.885 |
| Word2Vec (свой) + LogReg | 40 000 | ≈0.87 (на всём тесте 0.866) |
| GloVe + LogReg | 40 000 | ≈0.80 (на всём тесте 0.799) |
| MiniLM + LogReg | 40 000 | 0.812 |

Наш fine-tune на **в 10 раз меньшем** объёме данных почти догнал TF-IDF и правильно разобрал все «сложные» отзывы с отрицаниями и контрастом. Чтобы обогнать TF-IDF, нужно больше данных и эпох (на всём train обычно получают ~0.92–0.93) — но для этого уже нужен GPU.

Главная мысль: **трансформер-эмбеддинги — не волшебная кнопка**. Если модель не обучена под задачу, хорошо настроенный TF-IDF может её догнать или обогнать. Максимум качества даёт **fine-tuning** — дообучение трансформера на своих данных.